In [0]:
--Tabla 1: silver_matchs-------------------------------------------------------------------
--DATOS DEL MUNDIAL----------
WITH parsed_matches AS (
    SELECT
        name, 
        league,
        date,
        get_json_object(match, '$[0].homeTeam.name') AS local_team,
        get_json_object(match, '$[0].awayTeam.name') AS away_team,
        get_json_object(match, '$[0].location.name') AS stadium,
        get_json_object(match, '$[0].startDate') AS start_date,
        get_json_object(match, '$[0].endDate') AS end_date,
        sub_event.name AS event,
        sub_event.attendee.name AS event_team,
        link
    FROM 
        workspace.futbol.bronze_matchs
    LATERAL VIEW EXPLODE(FROM_JSON(get_json_object(match, '$[0].subEvent'), 'array<struct<`@type`:string, name:string, startDate:string, location:struct<`@type`:string, name:string>, attendee:struct<`@type`:string, name:string>>>')) exploded AS sub_event
),
cleaned_matches AS (
    SELECT
        REPLACE(substring_index(link, '-', -1), "/", "") AS match_id,
        DECODE(ENCODE(name, 'ISO-8859-1'), 'UTF-8') AS name,
        DECODE(ENCODE(league, 'ISO-8859-1'), 'UTF-8') AS league,
        date,
        DECODE(ENCODE(local_team, 'ISO-8859-1'), 'UTF-8') AS home_team,
        DECODE(ENCODE(away_team, 'ISO-8859-1'), 'UTF-8') AS away_team,
        DECODE(ENCODE(stadium, 'ISO-8859-1'), 'UTF-8') AS stadium,
        start_date,
        end_date,
        DECODE(ENCODE(event, 'ISO-8859-1'), 'UTF-8') AS event,
        DECODE(ENCODE(event_team, 'ISO-8859-1'), 'UTF-8') AS event_team
    FROM
        parsed_matches
)
SELECT
    match_id,
    name,
    league,
    date,
    start_date,
    end_date,
    home_team,
    away_team,
    SUM(CASE WHEN event IS NOT NULL AND event LIKE '%Gol %' AND event NOT LIKE '%VAR: %' AND event_team = home_team THEN 1 ELSE 0 END) AS home_goals,
    SUM(CASE WHEN event IS NOT NULL AND event LIKE '%Gol %' AND event NOT LIKE '%VAR: %' AND event_team = away_team THEN 1 ELSE 0 END) AS away_goals,
    CONCAT(home_goals, '-', away_goals) AS score
FROM
    cleaned_matches
GROUP BY
    match_id,
    name,
    league,
    date,
    start_date,
    end_date,
    home_team,
    away_team
ORDER BY
    date DESC
;


--Tabla 2: silver_match_events-------------------------------------------------------------------
--DATOS DEL MUNDIAL----------
WITH clean_events AS (
    SELECT
        REPLACE(substring_index(get_json_object(match, '$[0].url'), '-', -1), "/", "") AS match_id,
        CAST(get_json_object(sub_event_json, '$.startDate') AS TIMESTAMP) AS event_time,
        DECODE(ENCODE(get_json_object(sub_event_json, '$.attendee.name'), 'ISO-8859-1'), 'UTF-8') AS event_team,
        DECODE(ENCODE(get_json_object(sub_event_json, '$.name'), 'ISO-8859-1'), 'UTF-8') AS description,
        CASE
            WHEN get_json_object(sub_event_json, '$.name') IS NULL THEN 'OTHER'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Gol%' THEN 'GOL'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Amonest%' THEN 'TARJETA AMARILLA'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Expulsi%' THEN 'TARJETA ROJA'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Sale%' THEN 'CAMBIO'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'VAR:%' THEN 'VAR'
            ELSE 'OTHER'
        END AS event_type,
        date AS match_date,
        DECODE(ENCODE(name, 'ISO-8859-1'), 'UTF-8') AS match_name,
        DECODE(ENCODE(league, 'ISO-8859-1'), 'UTF-8') AS league
    FROM 
        workspace.futbol.bronze_matchs
    LATERAL VIEW EXPLODE(from_json(get_json_object(match, '$[0].subEvent'), 'array<string>')) exploded AS sub_event_json
    WHERE 
        get_json_object(match, '$[0].subEvent') IS NOT NULL
)
SELECT 
    DISTINCT
        match_id,
        event_time,
        event_team,
        description,
        event_type,
        match_date,
        match_name,
        league
FROM 
    clean_events
ORDER BY
    event_time DESC
;

--Tabla 2: silver_match_events (CON LOS MINUTOS EXACTOS DE CADA EVENTO!!)-------------------------------------------------------------------
--DATOS DEL MUNDIAL----------
WITH clean_events AS (
    SELECT
        REPLACE(substring_index(get_json_object(match, '$[0].url'), '-', -1), "/", "") AS match_id,
        CAST(get_json_object(sub_event_json, '$.startDate') AS TIMESTAMP) AS event_time,
        DECODE(ENCODE(get_json_object(sub_event_json, '$.attendee.name'), 'ISO-8859-1'), 'UTF-8') AS event_team,
        DECODE(ENCODE(get_json_object(sub_event_json, '$.name'), 'ISO-8859-1'), 'UTF-8') AS description,
        CASE
            WHEN get_json_object(sub_event_json, '$.name') IS NULL THEN 'OTHER'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Gol%' THEN 'GOL'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Amonest%' THEN 'TARJETA AMARILLA'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Expulsi%' THEN 'TARJETA ROJA'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'Sale%' THEN 'CAMBIO'
            WHEN get_json_object(sub_event_json, '$.name') LIKE 'VAR:%' THEN 'VAR'
            ELSE 'OTHER'
        END AS event_type,
        date AS match_date,
        DECODE(ENCODE(name, 'ISO-8859-1'), 'UTF-8') AS match_name,
        DECODE(ENCODE(league, 'ISO-8859-1'), 'UTF-8') AS league
    FROM 
        workspace.futbol.bronze_matchs
    LATERAL VIEW EXPLODE(from_json(get_json_object(match, '$[0].subEvent'), 'array<string>')) exploded AS sub_event_json
    WHERE 
        get_json_object(match, '$[0].subEvent') IS NOT NULL
),
events_with_base_times AS (
    SELECT 
        *,
        -- Buscamos el timestamp de inicio de cada tiempo por cada partido usando funciones de ventana
        MAX(CASE WHEN description LIKE 'Comienzo del partido%' THEN event_time END) OVER(PARTITION BY match_id) AS start_match_time,
        MAX(CASE WHEN description LIKE 'Comienzo del 2° Tiempo%' THEN event_time END) OVER(PARTITION BY match_id) AS start_second_half_time
    FROM 
        clean_events
)
SELECT 
    DISTINCT
        match_id,
        event_time,
        match_minute,
        event_minute,
        event_team,
        description,
        event_type,
        match_date,
        match_name,
        league
FROM (
    SELECT 
        match_id,
        event_time,
        -- Calculamos el minuto transcurrido en base al tiempo actual y los puntos de referencia
        CASE 
            -- Si ya empezó el 2do tiempo, calculamos los minutos desde ese punto y sumamos 46
            WHEN start_second_half_time IS NOT NULL AND event_time >= start_second_half_time THEN 
                FLOOR((UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(start_second_half_time)) / 60) + 46
            -- Si solo ha empezado el partido, calculamos desde el inicio y sumamos 1
            WHEN start_match_time IS NOT NULL AND event_time >= start_match_time THEN 
                FLOOR((UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(start_match_time)) / 60) + 1
            ELSE NULL 
        END AS match_minute, -- Nuevo campo con el conteo de minutos
        -- Minuto exacto desde el inicio del partido para cada evento
        CASE
            WHEN start_match_time IS NOT NULL AND event_time >= start_match_time THEN FLOOR((UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(start_match_time)) / 60) + 1
            ELSE NULL
        END AS event_minute, -- Minuto exacto desde el inicio del partido
        event_team,
        description,
        event_type,
        match_date,
        match_name,
        league
    FROM 
        events_with_base_times
)
WHERE
    match_minute IS NOT NULL
ORDER BY
    match_id,
    event_time DESC
;


--Tabla 3: silver_players-------------------------------------------------------------------
--DATOS DEL MUNDIAL----------
CREATE OR REPLACE TABLE workspace.futbol.silver_players AS

WITH cleaned_players AS (
    SELECT
        REPLACE(substring_index(parsed_match[0].url, '-', -1), '/', '') AS match_id,
        DECODE(ENCODE(team.name, 'ISO-8859-1'), 'UTF-8') AS team_name,
        DECODE(ENCODE(team_players.name, 'ISO-8859-1'), 'UTF-8') AS player_name,
        DECODE(ENCODE(team_players.roleName, 'ISO-8859-1'), 'UTF-8') AS player_position,
        -- Determina si el jugador fue titular o suplente
        CASE 
            WHEN team_players.roleName LIKE '%Substitute%' OR team_players.roleName LIKE '%Bench%' THEN 'SUPLENTE'
            ELSE 'TITULAR'
        END AS starter_status
    FROM (
        SELECT from_json(match, 'array<struct<`@context`:string, `@type`:string, url:string, `@graph`:array<struct<`@type`:string, name:string, athlete:array<struct<`@type`:string, name:string, roleName:string>>>>>>') AS parsed_match
        FROM workspace.futbol.bronze_matchs
    )
        LATERAL VIEW EXPLODE(parsed_match[1].`@graph`) t AS team
        LATERAL VIEW EXPLODE(team.athlete) p AS team_players
    WHERE
        team.`@type` = 'SportsTeam'
),
deduplicated AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY match_id, team_name, player_name ORDER BY  player_position) AS rn
    FROM
        cleaned_players
)
SELECT
    match_id,
    team_name,
    player_name,
    player_position,
    starter_status
FROM 
    deduplicated
WHERE 
    rn = 1
;






SELECT * FROM workspace.futbol.silver_players WHERE team_name = 'Racing'
SELECT * FROM workspace.futbol.silver_match_events WHERE event_type = 'GOL' ORDER BY event_time DESC



SELECT
m.away_team,
m.home_team,
m.league,
e.description,
e.event_type,
e.event_time AS date
FROM
workspace.futbol.silver_matchs m
JOIN workspace.futbol.silver_match_events e
ON m.match_id = e.match_id
ORDER BY date DESC




-- Query: Jugadores de Racing en 2026 ordenados por partido y fecha
SELECT 
    m.match_id,
    m.start_date AS match_date,
    m.name AS match_name,
    m.league,
    m.home_team,
    m.away_team,
    p.player_name,
    p.player_position,
    p.starter_status
FROM workspace.futbol.silver_matchs m
JOIN workspace.futbol.silver_players p 
    ON m.match_id = p.match_id
WHERE 
    p.team_name LIKE '%Racing%'
    -- AND YEAR(m.start_date) = 2026
ORDER BY 
    m.start_date DESC,
    m.match_id,
    p.starter_status,
    p.player_name;